# 03 · Embedding and storage

Stage 3 (`app/ingestion/embed.py`, `app/ingestion/store.py`) with the **real** models:
BGE-M3 (dense, 1024 numbers, meaning) and BM25 (sparse, exact words). Chapter XIV of the
Sales Tax Rules (pages 105-113) is embedded and stored in an in-memory Qdrant. The excerpt is
a 9-page PDF of its own, so inside this notebook its pages are numbered 1-9.

In [1]:
# Run from the project root so app/ imports and corpus/ paths work.
import os, sys, time, warnings
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
# Silence per-item progress bars from the model libraries so outputs stay readable.
os.environ["TQDM_DISABLE"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
warnings.filterwarnings("ignore")

import pandas as pd
pd.set_option("display.max_colwidth", 90)
print("project root:", Path.cwd().name)

project root: fbr-di-assistant


In [2]:
import asyncio
import pymupdf
from app.ingestion.caption import FakeCaptioner
from app.ingestion.chunk import chunk_blocks
from app.ingestion.extract import Extractor
from app.ingestion.validate import validate_file

with pymupdf.open("corpus/legal/sales_tax_rules_2006_upto_2026-07-31.pdf") as source, pymupdf.open() as subset:
    subset.insert_pdf(source, from_page=104, to_page=112)  # Chapter XIV, pages 105-113
    data = subset.tobytes()
blocks = (await Extractor(FakeCaptioner()).extract(validate_file(data, "chapter_xiv.pdf"))).blocks
chunks = chunk_blocks(blocks, "The Sales Tax Rules, 2006")
print(len(chunks), "chunks;", sum(c.chunk_type == "legal_rule" for c in chunks), "of them single rules")

32 chunks; 31 of them single rules


## 1. Load the models once

In the API this happens once in the FastAPI lifespan, never per request.

In [3]:
from app.ingestion.embed import BGEM3Embedder, Bm25Embedder

t0 = time.time()
dense_model, sparse_model = BGEM3Embedder(), Bm25Embedder()
print(f"models loaded in {time.time() - t0:.1f}s")

models loaded in 19.4s


## 2. Embed: dense and sparse at the same time

Measured on this laptop's CPU: about 1-1.5 s per chunk with all threads and batch size 1, versus
4.5 s with the library defaults (half the threads, batch 16 padded to its longest text). Long
chunks cost more: the model processes the longest first, at up to ~3 s each.

In [4]:
texts = [c.embed_text for c in chunks]
t0 = time.time()
dense, sparse = await asyncio.gather(dense_model.embed_documents(texts), sparse_model.embed_documents(texts))
took = time.time() - t0
print(f"{len(texts)} chunks in {took:.1f}s -> {took / len(texts):.2f}s per chunk")
print(f"dense: {len(dense[0])} numbers, length {sum(v * v for v in dense[0]) ** 0.5:.3f} (normalised)")
print(f"sparse: first chunk has {len(sparse[0].indices)} distinct words out of a vocabulary of millions")

32 chunks in 46.9s -> 1.47s per chunk
dense: 1024 numbers, length 1.000 (normalised)
sparse: first chunk has 31 distinct words out of a vocabulary of millions


## 3. Dense meaning versus sparse exact words

For each question: the top 3 chunks by cosine similarity (dense) and by BM25 overlap (sparse).
The paraphrased questions share almost no words with the rule; the code question is the opposite.

In [5]:
import numpy as np

matrix = np.array(dense)
def top_dense(q, k=3):
    scores = matrix @ np.array(q)  # dot product = cosine, since vectors are normalised
    return [(chunks[i].section_path[-1], round(float(scores[i]), 2)) for i in np.argsort(-scores)[:k]]
def top_sparse(q, k=3):
    weights = dict(zip(q.indices, q.values))
    scores = [sum(weights.get(i, 0) * v for i, v in zip(s.indices, s.values)) for s in sparse]
    order = np.argsort(-np.array(scores))[:k]
    return [(chunks[i].section_path[-1], round(float(scores[i]), 2)) for i in order]

questions = ["We won't finish integration on time. Can we get more time, and how much?",
             "What happens if we tamper with the integrated system?",
             "Rule 150XF"]
rows = []
for q in questions:
    qd, qs = await asyncio.gather(dense_model.embed_query(q), sparse_model.embed_query(q))
    rows.append({"question": q, "dense top 3": top_dense(qd), "bm25 top 3": top_sparse(qs)})
pd.DataFrame(rows)

,question,dense top 3,bm25 top 3
0,"We won't finish integration on time. Can we get more time, and how much?","[(Rule 150V, 0.6), (Rule 150XA, 0.5), (Rule 150R, 0.49)]","[(Rule 150V, 3.74), (Rule 150T, 3.34), (Rule 150XQ, 3.33)]"
1,What happens if we tamper with the integrated system?,"[(Rule 150X, 0.64), (Rule 150XQ, 0.54), (Rule 150XA, 0.53)]","[(Rule 150XA, 4.95), (Rule 150X, 4.77), (Rule 150R, 3.81)]"
2,Rule 150XF,"[(Rule 150XM, 0.61), (Rule 150XE, 0.6), (Rule 150XG, 0.6)]","[(Rule 150XF, 3.77), (Rule 150XE, 3.4), (Rule 150XN, 1.96)]"


**Read the last row:** to a meaning model "150XF", "150XM" and "150XE" look almost the same,
so dense search misses; BM25 matches the exact token. The first two rows show the reverse
strength. Hence hybrid retrieval (dense + BM25, fused) in Phase 5. Note that these BM25 scores
lack the IDF weight, which Qdrant adds at search time.

## 4. Store, then prove a delete leaves nothing

Each point carries both vectors and the full payload a citation needs.

In [6]:
import uuid
from qdrant_client import AsyncQdrantClient
from app.ingestion.store import DocumentMeta, count_document_points, delete_document_points, ensure_collection, upsert_chunks

client = AsyncQdrantClient(location=":memory:")
meta = DocumentMeta(doc_id=uuid.uuid4(), owner_id=uuid.uuid4(), filename="chapter_xiv.pdf", authority="law", version_date="2026-07-31")
await ensure_collection(client, "demo")
print("stored:", await upsert_chunks(client, chunks, dense, sparse, meta, name="demo"))
records, _ = await client.scroll("demo", limit=100, with_payload=True)
point = next(r for r in records if r.payload["section_path"][-1:] == ["Rule 150V"])
payload = dict(point.payload)
payload["text"] = payload["text"][:80] + "..."
payload

stored: 32


{'chunk_id': '6da1e089-0eb4-5243-88bd-d4e08a15eea4',
 'chunk_index': 10,
 'owner_id': '9afa61ef-f7fb-49d0-90e5-d0e2b812a992',
 'doc_id': 'e8220193-3843-4032-aadd-a9e7760c94d3',
 'filename': 'chapter_xiv.pdf',
 'page_start': 4,
 'page_end': 4,
 'section_path': ['CHAPTER XIV PROCEDURE FOR LICENSING, ISSUANCE OF ELECTRONIC SALES TAX INVOICES AND INTEGRATION OF REGISTERED PERSONS',
  'Rule 150V'],
 'chunk_type': 'legal_rule',
 'authority': 'law',
 'version_date': '2026-07-31',
 'superseded': False,
 'amended_by': [],
 'ocr': False,
 'suspicious': False,
 'content_hash': 'e38748dbe4fefaf2948639a5de8cf000b3d0bcd7046d14a47a118d5ceca26add',
 'text': '150V. Extension in due date of Integration. — The Commissioner Inland Revenue ha...'}

In [7]:
print("points before delete:", await count_document_points(client, meta.doc_id, name="demo"))
await delete_document_points(client, meta.doc_id, meta.owner_id, name="demo")
print("points after delete :", await count_document_points(client, meta.doc_id, name="demo"))
await client.close()

points before delete: 32
points after delete : 0
